# Solutions · 07 · Lead, lag and loop shaping

Worked solutions to the exercises of [notebook 07](../notebooks/07_lead_lag_and_loop_shaping.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, design as dg, frequency as fr, response as rs, models, polynomials as pl
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Design a lead-lag for 1/(s (s + 1)): crossover 3 rad/s, phase margin 50 degrees, and a velocity error constant Kv >= 50. Report the final margins and the ramp error.

In [2]:
P = sy.tf([1.0], [1.0, 1.0, 0.0])
lead_d = dg.design_lead(P, w_c=3.0, pm_target_deg=50.0)
C_lead = lead_d["controller"]
Kv_now = sy.series(C_lead, P).num[-1] / sy.series(C_lead, P).den[-2]          # velocity constant = lim s L(s)
beta = 50.0 / Kv_now
C_lag = dg.lag(beta, 3.0 / 10.0)
C = sy.series(C_lag, C_lead)
L = sy.series(C, P)
m = fr.margins(L)
Kv = L.num[-1] / L.den[-2]
t = np.linspace(0, 20, 4001)
_, y_ramp, _ = rs.lsim(sy.feedback(L), t, t)
print(f"lead alone: Kv = {Kv_now:.2f}; lag with beta = {beta:.2f} (zero a decade below the crossover) -> Kv = {Kv:.1f}, crossover {m['w_gc']:.2f} rad/s, phase margin {m['phase_margin']:.1f} deg, ramp error {t[-1] - y_ramp[-1]:.4f} (1/Kv = {1 / Kv:.4f})")

lead alone: Kv = 4.78; lag with beta = 10.47 (zero a decade below the crossover) -> Kv = 50.0, crossover 3.01 rad/s, phase margin 49.8 deg, ramp error 0.0228 (1/Kv = 0.0200)


The lead sets the crossover and the margin; the lag multiplies the low-frequency gain by β without moving
them, because its zero sits a decade below the crossover. The velocity error constant rises from 5 to 50 and
the ramp error falls to 1/K_v = 0.02, at a cost of a few degrees of phase margin - the classical lead-lag
recipe.

## Exercise 2

Repeat the lead design of the motor position loop with the sensor noise in mind: add white noise of 0.01 rad rms to the measurement in a simulation (use `response.lsim` on CS with a noise input) and tabulate the control-signal noise against the target crossover.

In [3]:
mot = models.dc_motor(position=True)["system"].to_tf()
rng = np.random.default_rng(0)
t = np.linspace(0, 2, 2001)
noise = 0.01 * rng.normal(size=len(t))
rows = []
for wc in (2.0, 5.0, 10.0, 20.0):
    d = dg.design_lead(mot, w_c=wc, pm_target_deg=50.0)
    cl = dg.closed_loop(mot, d["controller"])
    _, u_noise, _ = rs.lsim(cl["CS"], noise, t)                      # measurement noise n enters the control through -CS n (sign irrelevant for the rms)
    rows.append((wc, d["K"], np.std(u_noise), np.max(np.abs(fr.freqresp(cl["CS"], np.logspace(-1, 4, 2000))))))
print(pd.DataFrame(rows, columns=["target w_c", "gain K", "control noise rms (V)", "peak |CS|"]).to_string(index=False, float_format="%.3f"))

 target w_c  gain K  control noise rms (V)  peak |CS|
      2.000  19.729                  0.423     54.779
      5.000  40.690                  5.591    556.766
     10.000 127.776                 41.156   4068.972
     20.000 410.668                509.602  49182.300


Measurement noise reaches the actuator through CS, whose high-frequency gain is the controller's: each
doubling of the crossover multiplies the control-signal noise by about ten here (two cascaded leads, each
with a growing 1/α), until at 20 rad/s a 0.01 rad sensor noise produces hundreds of volts of actuator
chatter - far beyond the 12 V supply. Bandwidth is paid in noise amplification as
surely as in control effort; a lead's high-frequency gain 1/α is the price tag.

## Exercise 3

**Implement it yourself:** write the lead design procedure (read the plant phase at w_c, compute the needed lead, build the compensator, set the gain) and reproduce `design.design_lead` for the DC motor position loop at w_c = 10 rad/s.

In [4]:
P = models.dc_motor(position=True)["system"].to_tf()
w_c, pm_target, extra = 10.0, 50.0, 5.0
w_grid = np.logspace(np.log10(w_c) - 4, np.log10(w_c), 2000)
phase_now = np.degrees(np.unwrap(np.angle(fr.freqresp(P, w_grid))))[-1]
needed = pm_target - (180.0 + phase_now) + extra
n_leads = 1 if needed <= 65 else int(np.ceil(needed / 60.0))
phi = np.radians(needed / n_leads)
alpha = (1 - np.sin(phi)) / (1 + np.sin(phi)); T = 1 / (w_c * np.sqrt(alpha))
C = sy.tf([1.0], [1.0])
for _ in range(n_leads):
    C = sy.series(C, sy.tf([T, 1.0], [alpha * T, 1.0]))
K = 1 / abs(fr.freqresp(sy.series(C, P), w_c))
C = sy.tf(K * C.num, C.den)
m = fr.margins(sy.series(C, P))
lib = dg.design_lead(P, w_c, pm_target)
print(f"plant phase at {w_c} rad/s: {phase_now:.1f} deg -> lead needed {needed:.1f} deg in {n_leads} stage(s); result: crossover {m['w_gc']:.3f}, phase margin {m['phase_margin']:.1f} deg; library: {lib['margins']['w_gc']:.3f}, {lib['margins']['phase_margin']:.1f} deg")

plant phase at 10.0 rad/s: -213.7 deg -> lead needed 88.7 deg in 2 stage(s); result: crossover 10.000, phase margin 55.0 deg; library: 10.000, 55.0 deg


The procedure in five lines: unwrap the plant's phase up to the target crossover, compute the missing lead
(with a few degrees of allowance), split it over two stages when one cannot supply it, build the leads with
their maximum phase at the crossover, and scale the gain to unity loop gain there. The result matches the
library's to the digit.